# 🌊 BlueGuard: Ocean Waste Classification

## Overview
This notebook presents a binary image classification pipeline designed to distinguish
between clean ocean surfaces and ocean areas polluted with waste.

The dataset contains two classes:
- **ocean_clean** — images of clean, unpolluted ocean
- **ocean_trash** — images of ocean surfaces containing visible waste and debris

## Approach
We fine-tune a pretrained **EfficientNet-B0** model using transfer learning from
ImageNet weights. The full dataset is split into train/validation/test sets (80/10/10)
and trained using the **AdamW** optimizer with **Cosine Annealing** learning rate scheduling.

To maximize GPU utilization, **DataParallel** is applied across dual NVIDIA T4 GPUs
available on the Kaggle accelerator environment.

## Goals
- Build a robust binary classifier for ocean waste detection
- Apply proper data augmentation to improve generalization
- Evaluate model performance using precision, recall, F1-score, and confusion matrix

## Cell 1 — GPU Check

In this cell, we verify the PyTorch version and detect the number of available GPUs.
We set the `device` variable which will be used throughout the notebook to move
tensors and models to the correct hardware (GPU or CPU).

In [ ]:
import torch

print(f"Pytorch version: {torch.__version__}")
print(f"GPU sayısı: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nKullanılan device: {device}")

## Cell 2 — Import Libraries

We import all required libraries for this project:
- **os, random, numpy, pandas** — general utilities
- **PIL** — image loading and processing
- **sklearn** — evaluation metrics (classification report, confusion matrix)
- **seaborn, matplotlib** — visualization
- **torch, torchvision** — deep learning framework and pretrained models

In [ ]:
import os
import random
import numpy as np
import pandas as od
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

## Cell 3 — Reproducibility

We fix the random seed across Python, NumPy, and PyTorch to ensure that results
are reproducible across different runs. `cudnn.deterministic = True` disables
non-deterministic GPU operations.

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("Seed ayarlandı.")

## Cell 4 — Configuration

All hyperparameters and paths are defined in a single `CONFIG` dictionary.
This makes it easy to modify settings without hunting through the code.
Key parameters:
- `img_size` — input resolution for the model (224x224)
- `batch_size` — 64 samples per batch, optimized for dual T4 GPUs
- `epochs` — total number of training cycles
- `lr` — initial learning rate for the AdamW optimizer
- `train/val/test_ratio` — 80/10/10 split applied to the full dataset

In [ ]:
CONFIG = {
    "data_dir" : "/kaggle/input/datasets/hassnainzaidi/domain",
    "train_dir": 224,
    "batch_size": 64,
    "epochs" : 20,
    "lr" : 1e-4,
    "num_classes" : 2,
    "num_workers" : 4,
    "model_name" : "efficientnet_b0",
    "save_path" : "/kaggle/working/best_model.pth",
    "classes" : ["ocean_clean", "ocean_trash"],
    "train_ratio" : 0.80,
    "val_ratio" : 0.10,
    "test_ratio" : 0.10
}

print("Config hazır: ", CONFIG)

## Cell 5 — Dataset Structure Check

We verify that both class folders (`ocean_clean` and `ocean_trash`) exist
in the data directory and print the number of images found in each class.
This helps catch path errors early before training begins.

In [ ]:
for cls in CONFIG["classes"]:
    cls_path = os.path.join(CONFIG["data_dir"], cls)
    if os.path.exists(cls_path):
        count = len(os.listdir(cls_path))
        print(f"{cls}: {count} görsel")
    else:
        print(f"UYARI: {cls_path} bulunamadı!")

## Cell 6 — Transforms

We define separate image transformation pipelines for training and validation/test sets.

**Training transforms** include random augmentations (horizontal/vertical flip,
rotation, color jitter) to improve generalization and reduce overfitting.

**Validation/Test transforms** only resize and normalize — no augmentation —
to get a clean, unbiased evaluation.

ImageNet mean and std values are used for normalization since we use a pretrained model.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CONFIG["train_dir"], CONFIG["train_dir"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.3225])
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG["train_dir"], CONFIG["train_dir"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.3225])
])

print("Transformlar tanımlandı.")

## Cell 7 — Custom Dataset Class

We define a custom `OceanDataset` class that inherits from PyTorch's `Dataset`.
It scans each class folder, collects all valid image paths with their labels,
and loads images on demand during training.

- `__len__` returns the total number of samples
- `__getitem__` loads an image, converts it to RGB, applies the transform, and returns (image, label)

In [ ]:
class OceanDataset(Dataset):
    def __init__(self,root_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}

        for cls in classes:
            cls_folder = os.path.join(root_dir, cls)
            if not os.path.exists(cls_folder):
                print(f"UYARI: {cls_folder} bulunamadı, atlanıyor.")
                continue
            for fname in os.listdir(cls_folder):
                if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    self.samples.append(
                        (os.path.join(cls_folder, fname), self.class_to_idx[cls])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

print("OceanDataset sınıfı tanımlandı.")

## Cell 8 — DataLoaders

We split the full dataset into train/val/test subsets using random index shuffling
with a fixed seed (42) for reproducibility.

Critically, we create **two separate dataset objects** — one with `train_transform`
and one with `val_transform` — then use `Subset` to assign the correct transform
to each split. This prevents augmentation from leaking into validation and test sets.

`DataLoader` wraps each subset with batching, shuffling (train only), and multi-worker
parallel loading. `pin_memory=True` speeds up GPU data transfer.

In [ ]:
from torch.utils.data import random_split

full_dataset = OceanDataset(CONFIG["data_dir"], CONFIG["classes"], transform=None)
total = len(full_dataset)

train_size = int(total * CONFIG["train_ratio"])
val_size = int(total * CONFIG["val_ratio"])
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Her split'e kendi transform'unu uygula
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform
test_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Toplam: {total} | Train: {train_size} | Val: {val_size} | Test: {test_size}")

## Cell 9 — Sample Visualization

We randomly sample images from the training set and display them with their
class labels. The normalization is reversed before display so images appear
in their natural colors. This is a quick sanity check to confirm that images
and labels are correctly loaded.

In [ ]:
def show_samples(dataset, classes, n=6):
    fig, axes = plt.subplots(1, n, figsize=(15, 4))
    indices = random.sample(range(len(dataset)), n)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    for ax, idx in zip(axes, indices):
        img, label = dataset[idx]
        img = img.numpy().transpose(1, 2, 0)
        img = std * img + mean
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(classes[label])
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, CONFIG["classes"])

## Cell 10 — Model (EfficientNet + DataParallel)

We load a pretrained `EfficientNet-B0` from torchvision and replace its
classification head with a new `Linear` layer matching our 2-class problem.
Using ImageNet pretrained weights gives us a strong feature extractor from the start.

If more than one GPU is detected, we wrap the model with `nn.DataParallel`
which automatically splits each batch across all available GPUs and merges
the gradients — effectively doubling our throughput on dual T4 GPUs.

In [ ]:
%%capture
def build_model(num_classes, num_gpus):
    model = models.efficientnet_b0(weights="IMAGENET1K_V1")

    # Classifier başını değiştir
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes)
    )

    if num_gpus > 1:
        print(f"DataParallel aktif - {num_gpus} GPU kullanılıyor.")
        model = nn.DataParallel(model)

    model = model.to(device)
    return model

num_gpus = torch.cuda.device_count()
model = build_model(CONFIG["num_classes"], num_gpus)
print(model)


## Cell 11 — Loss, Optimizer, Scheduler

- **CrossEntropyLoss** — standard loss function for multi-class classification
- **AdamW** — Adam optimizer with decoupled weight decay, better generalization than vanilla Adam
- **CosineAnnealingLR** — gradually reduces the learning rate following a cosine curve,
  helping the model settle into a better minimum towards the end of training

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
    eta_min=1e-6
)

print("Loss, Optimizer ve Schedule hazır.")

## Cell 12 — Train & Validation Functions

We define two separate functions:

**`train_one_epoch`** — sets the model to train mode, runs forward pass,
computes loss, backpropagates gradients, and updates weights for each batch.

**`validate`** — sets the model to eval mode and runs inference without
gradient computation (`torch.no_grad()`). Collects all predictions and
ground truth labels for metric calculation.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct +=(preds == labels).sum().item()
        total += imgs.size(0)

    return total_loss / total, correct / total

def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels= [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct +=(preds == labels).sum().item()
            total += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels
print("Train & Validate fonksiyonları hazır.")

## Cell 13 — Training Loop

The main training loop iterates over all epochs. For each epoch:
1. Train for one full pass over the training set
2. Evaluate on the validation set
3. Step the learning rate scheduler
4. Log metrics to `history` for later plotting
5. Save the model weights if validation accuracy improves (best model checkpoint)

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch [{epoch:02d}/{CONFIG['epochs']}]"
         f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
         f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CONFIG["save_path"])
        print(f" Model kaydedildi - Val Acc: {best_val_acc:.4f}")

print(f"\nEğitim tamamlandı. En iyi Val Acc: {best_val_acc:.4f}")
    

## Cell 14 — Loss & Accuracy Plots

We plot training and validation loss/accuracy curves across all epochs.
These curves help diagnose issues such as overfitting (val loss rising while
train loss drops) or underfitting (both curves plateauing at high loss).
The plots are saved to `/kaggle/working/` for download.

In [ ]:
epochs_range = range(1, CONFIG["epochs"] + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (14, 5))

ax1.plot(epochs_range, history["train_loss"], label= "Train Loss")
ax1.plot(epochs_range, history["val_loss"], label= "Val Loss")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(epochs_range, history["train_acc"], label= "Train Acc")
ax2.plot(epochs_range, history["val_acc"], label= "Val Acc")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()

## Cell 15 — Load Best Model & Evaluate

We reload the best saved checkpoint (highest val acc during training) and
run it on the validation set. We then print a full `classification_report`
showing per-class precision, recall, and F1-score, giving a detailed view
of model performance beyond simple accuracy.

In [ ]:
model.load_state_dict(torch.load(CONFIG["save_path"], map_location=device))
_, val_acc, all_preds, all_labels = validate(model, val_loader, criterion)

print(f"Yüklenen model Val Acc: {val_acc:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=CONFIG["classes"]))

## Cell 16 — Confusion Matrix

We visualize the confusion matrix as a heatmap using seaborn.
Each cell shows the count of samples: correct predictions appear on the diagonal,
misclassifications appear off-diagonal. This makes it easy to see if the model
is biased towards one class.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
           xticklabels=CONFIG["classes"],
           yticklabels=CONFIG["classes"])
plt.title("Confusion Matrix")
plt.ylabel("Gerçek")
plt.xlabel("Tahmin")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()

## Cell 17 — Wrong Predictions

We iterate over the validation set and collect images where the model's prediction
did not match the ground truth label. We display these misclassified samples
with both the correct label (GT) and the predicted label, which helps us
qualitatively understand what kinds of images the model struggles with.

In [ ]:
def show_wrong_predictions(dataset, model, classes, n=8):
    model.eval()
    wrong_imgs, wrong_labels, wrong_preds = [], [], []
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
    with torch.no_grad():
        for imgs, labels in loader:
            outputs = model(imgs.to(device))
            preds = outputs.argmax(dim=1).cpu()
            mask = preds != labels
            wrong_imgs.extend(imgs[mask])
            wrong_labels.extend(labels[mask])
            wrong_preds.extend(preds[mask])
            if len(wrong_imgs) >= n:
                break

    fig, axes = plt.subplots(1, min(n, len(wrong_imgs)), figsize=(15, 4))
    for ax, img, lbl, pred in zip(axes, wrong_imgs[:n], wrong_labels[:n], wrong_preds[:n]):
        img = img.numpy().transpose(1, 2, 0)
        img = std * img + mean
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"GT: {classes[lbl]}\nPred: {classes[pred]}", color="red", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/wrong_predictions.png", dpi=150)
    plt.show()

show_wrong_predictions(val_dataset, model, CONFIG["classes"])

# 📊 Conclusion

## Results Summary

The fine-tuned EfficientNet-B0 model achieved outstanding performance on the
BlueGuard Ocean Waste Classification dataset:

| Metric        | ocean_clean | ocean_trash | Overall  |
|---------------|-------------|-------------|----------|
| Precision     | 1.00        | 1.00        | 1.00     |
| Recall        | 1.00        | 1.00        | 1.00     |
| F1-Score      | 1.00        | 1.00        | 1.00     |
| **Val Acc**   | —           | —           | **99.87%** |

## Key Takeaways

- **Transfer learning** from ImageNet weights proved highly effective even for
  ocean-domain imagery, converging quickly within the first few epochs.
- **DataParallel** across dual T4 GPUs significantly reduced training time per epoch.
- Proper **transform isolation** between train and val/test splits was critical —
  applying augmentation to all splits caused artificially inflated accuracy.
- The model showed no signs of severe overfitting; train and validation metrics
  remained closely aligned throughout training.

## Potential Improvements

- **GradCAM visualization** — highlight which regions of the image the model
  focuses on when making predictions.
- **Larger backbone** — experimenting with EfficientNet-B3 or a Vision Transformer
  may yield marginal gains.
- **Test-Time Augmentation (TTA)** — averaging predictions over multiple augmented
  versions of the same image can further boost robustness.
- **Real-world deployment** — the model could be integrated into drone or satellite
  imagery pipelines for automated ocean waste monitoring.